# Microscope Inspection Notebook

Queries the live NIS-Elements microscope state using `nis_util.get_*` functions and derives the field of view.

In [1]:
import nis_util

NIS_EXE = r'C:\Program Files\NIS-Elements\nis_ar.exe'

## Camera format

Returns `(live_format, capture_format)` format strings (binning & bit depth).

In [2]:
fmt_live, fmt_capture = nis_util.get_camera_format(NIS_EXE)
print('Live:  ', fmt_live)
print('Capture:', fmt_capture)

Live:   FMT 1x1 (X-11056) 16-bit
Capture: FMT 1x1 (X-11056) 16-bit


## Stage position

Returns `(x, y, z0, z1)` — XY position and Z position of stage 0 (and the piezo/Z-stage 1 if present).

In [3]:
pos = nis_util.get_position(NIS_EXE)
x, y, z0, z1 = pos
print(f'X: {x}, Y: {y}')
print(f'Z (stage 0): {z0}')
if z1 is not None:
    print(f'Z (stage 1/piezo): {z1}')

X: 0.9, Y: -1.2
Z (stage 0): 421.275
Z (stage 1/piezo): 100.0


## Resolution & field of view

`get_resolution` returns `(xres, yres, pixel_size, magnification)`.

The FOV in microns follows from:

$$FOV_{x,y} = \frac{{xres_{y} \times pixel\_size}}{{magnification}}$$

(helper: `nis_util.get_fov_from_res`)

In [4]:
xres, yres, pixel_size, mag = nis_util.get_resolution(NIS_EXE)
print(f'Sensor: {xres} x {yres} pixels')
print(f'Pixel size: {pixel_size} um')
print(f'Magnification: {mag}x')

fov_x, fov_y = nis_util.get_fov_from_res((xres, yres, pixel_size, mag))
print(f'FOV: {fov_x:.1f} x {fov_y:.1f} um')

Sensor: 1024.0 x 1024.0 pixels
Pixel size: 13.0 um
Magnification: 20.0x
FOV: 665.6 x 665.6 um


## Calibration & rotation

Rotation/calibration matrix from `Get_CalibrationAngleMatrix`, and camera rotation angles.

> **Note on `get_cam_rotation`:** an earlier version called `CameraGet_Cam0Flip` / `CameraGet_Cam0Rotate180`, which do not exist in this NIS-Elements macro API — the unknown function aborts the whole macro at compile time, leaving the `.ini` result file empty (`KeyError: 'res'`). The function now uses only the documented `CameraGet_Rotate` / `Camera_RotateGet`. Flip/180° information is covered by the calibration matrix below (documented as the "rotation and flip" transformation).

In [5]:
a11, a12, a21, a22 = nis_util.get_rotation_matrix(NIS_EXE)
print('Calibration matrix:')
print(f'[{a11:+.5f}  {a12:+.5f}]')
print(f'[{a21:+.5f}  {a22:+.5f}]')

Calibration matrix:
[-0.99991  -0.01356]
[+0.01356  -0.99991]


In [ ]:
rotation, rotation2 = nis_util.get_cam_rotation(NIS_EXE)
print(f'Rotation (CameraGet_Rotate): {rotation} deg')
print(f'Rotation (Camera_RotateGet): {rotation2} deg')

## Optical configurations

Lists all available optical configurations. Use `nis_util.set_optical_configuration(NIS_EXE, name)` to switch.

In [ ]:
confs = nis_util.get_optical_confs(NIS_EXE)
for i, name in enumerate(confs):
    print(f'{i}: {name}')